# Drought-index insurance: a walkthrough

This notebook walks through the results of the synthetic workflow. It does **not** recompute anything: run `python run_pipeline.py` from the project root first, then run this notebook top to bottom.

**Question:** how well does a simple tiered contract, triggered by growing-season SPI-3, compensate farmers for drought-driven wheat yield losses?

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config

rain = pd.read_csv(config.RAW_DIR / "rainfall_monthly.csv", parse_dates=["date"])
spi = pd.read_csv(config.PROCESSED_DIR / "spi.csv", parse_dates=["date"])
events = pd.read_csv(config.RESULTS_DIR / "drought_events.csv", parse_dates=["start", "end"])
payouts = pd.read_csv(config.RESULTS_DIR / "payouts.csv")
metrics = pd.read_csv(config.RESULTS_DIR / "contract_metrics.csv")
FIG = config.FIGURES_DIR

## 1. Rainfall and SPI

Each region has its own seasonal cycle. SPI removes that seasonality: by construction it should have mean ≈ 0 and SD ≈ 1 in every region. The grey bands in the figure are the dry spells we injected on purpose; SPI-12 should sit well below −1 inside them.

In [ ]:
display(spi.groupby("region")[["spi_3", "spi_12"]].agg(["mean", "std"]).round(2))
display(Image(FIG / "fig1_rainfall_spi_Westridge.png", width=850))

## 2. Drought events (run theory, SPI-3 < −1)

The most severe events should line up with the injected dry spells. Many short one- or two-month events also appear: these are ordinary dry patches, the "noise" any real record contains.

In [ ]:
display(events.nlargest(8, "severity"))
display(Image(FIG / "fig2a_drought_events_timeline.png", width=850))

## 3. Growing-season SPI and yield

Yield responds to May–October rainfall (SPI-6 ending in October), but noise from other shocks keeps the relationship imperfect, as it is in real data.

In [ ]:
display(Image(FIG / "fig3_spi_vs_yield.png", width=950))

## 4. The contract

The index is the **lowest SPI-3** among the windows ending Jul, Aug, Sep and Oct. Payout tiers (fraction of the sum insured; the sum insured is 50% of expected yield):

In [ ]:
display(pd.DataFrame(config.PAYOUT_TIERS, columns=["SPI-3 at or below", "payout (share of sum insured)"]))
display(payouts.loc[payouts["paid"] | payouts["loss_year"],
                    ["region", "year", "trigger_spi3", "payout_frac", "loss_frac", "paid", "loss_year"]]
        .round(3).head(15))

## 5. How well do payouts match losses?

- **Hit rate**: share of loss years that received a payout.
- **False alarm ratio**: share of payouts made in years without a loss.
- **Basis risk (RMSE)**: typical gap between payout and loss, in fractions of expected yield.

In [ ]:
display(metrics[["region", "loss_years", "payouts", "hits", "misses", "false_alarms",
                 "hit_rate", "false_alarm_ratio", "basis_risk_rmse", "payout_loss_corr"]])
display(Image(FIG / "fig4a_payout_vs_loss_scatter.png", width=600))
display(Image(FIG / "fig4b_payout_loss_by_year.png", width=850))

## 6. What drives the basis risk here?

1. **Non-rainfall shocks**: yield noise (frost, pests, heat) causes losses the rainfall index cannot see, which shows up as missed losses.
2. **Index mismatch**: yield depends on the *whole-season* total (SPI-6), while the contract pays on the *worst 3-month window*. A dry August inside an otherwise normal season triggers a payout without a loss, which shows up as false alarms.
3. **Multiple windows raise trigger frequency**: taking the minimum of four overlapping SPI-3 values makes a payout more likely than any single window would.
4. **Tier steps**: payouts jump in steps while losses vary continuously, so even correct payouts rarely match the loss amount.

**Try it:** edit `src/config.py` (e.g. `TRIGGER_WINDOW_END_MONTHS = [8]`, stricter tiers, or `YIELD_NOISE_SD = 0.05`), re-run the pipeline, and see how the metrics change.